In [1]:
!pip install diffusers transformers accelerate torch --quiet

In [ ]:
import torch
from diffusers import StableDiffusionPipeline
from datetime import datetime
import os

In [3]:
def load_model(model_name="runwayml/stable-diffusion-v1-5"):
    """
    Load Stable Diffusion pretrained model
    """

    device = "cuda" if torch.cuda.is_available() else "cpu"

    print(f"Using device: {device}")

    pipe = StableDiffusionPipeline.from_pretrained(
        model_name,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32
    )

    pipe = pipe.to(device)

    return pipe

In [4]:
def generate_images(pipe, prompt, num_images=1, height=512, width=512, seed=None):
    """
    Generate images from text prompt
    """

    images = []

    if seed is not None:
        generator = torch.Generator().manual_seed(seed)
    else:
        generator = None

    for i in range(num_images):

        result = pipe(
            prompt,
            height=height,
            width=width,
            num_inference_steps=50,
            guidance_scale=7.5,
            generator=generator
        )

        images.append(result.images[0])

    return images

In [5]:
def save_images(images, output_dir="generated_images"):

    os.makedirs(output_dir, exist_ok=True)

    saved_paths = []

    for i, img in enumerate(images):

        filename = f"{output_dir}/image_{i}_{datetime.now().strftime('%H%M%S')}.png"
        img.save(filename)

        saved_paths.append(filename)

    return saved_paths

In [ ]:
def main():

    print("Loading Stable Diffusion Model...")
    pipe = load_model()

    prompt = input("Enter your image prompt: ")
    num_images = int(input("Number of images to generate: "))

    print("Generating images...")

    images = generate_images(pipe, prompt, num_images)

    paths = save_images(images)

    print("\nImages saved at:")
    for p in paths:
        print(p)

    for img in images:
        display(img)

main()